# 声明合并与模块扩充

学习目标：能读懂同名声明的合并规则，并用真实实现完成模块及全局扩充，识别只补类型却没有加载实现的错误。

前置知识：interface、类、函数重载、模块导入导出、declare、原型对象及类型和值。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict；另启用 verbatimModuleSyntax。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/19-declaration-merging/。

1. [main.ts](scripts/19-declaration-merging/main.ts)：配套实现与示例。
2. [gauge.ts](scripts/19-declaration-merging/gauge.ts)：配套实现与示例。
3. [gauge-double.ts](scripts/19-declaration-merging/gauge-double.ts)：配套实现与示例。
4. [global-build.ts](scripts/19-declaration-merging/global-build.ts)：配套实现与示例。
5. [missing-patch.ts](scripts/19-declaration-merging/missing-patch.ts)：配套实现与示例。
6. [tsconfig.json](scripts/19-declaration-merging/tsconfig.json)：本章独立项目配置。
7. [type-errors.ts](scripts/19-declaration-merging/type-errors.ts)、[tsconfig.errors.json](scripts/19-declaration-merging/tsconfig.errors.json)：单独检查的类型反例。

Step 1：检查本章正常示例的类型。

```bash
npm run check:19
```

Step 2：生成本章 JavaScript。

```bash
npm run build:19
```

Step 3：运行本章正常示例。

```bash
npm run run:19
# 正常退出；各段预期输出见代码注释。
```

同一文件的片段按正文顺序衔接，前文定义在后续片段中继续使用。直接运行完整项目；反例使用独立配置，不进入正常运行入口。

## 1 声明占据哪些名称空间

声明合并（declaration merging）把允许合并的同名声明作为一个实体检查；能否合并取决于声明种类及所在作用域。先分清名称用于类型、运行时值还是命名空间成员路径。

| 声明种类 | 中文名称／含义 | 类型 | 值 | 命名空间 |
| --- | --- | --- | --- | --- |
| interface | 接口声明 | 是 | 否 | 否 |
| type | 类型别名 | 是 | 否 | 否 |
| class | 类声明 | 是 | 是 | 否 |
| function | 函数声明 | 否 | 是 | 否 |
| enum | 枚举声明 | 是 | 是 | 否 |
| namespace | 命名空间声明 | 否 | 是 | 是 |

表中按普通非环境声明理解；仅包含类型或环境声明时不能据此推断一定生成运行时对象。下面接口 Stamp 与值 Stamp 可以同名，因为用途不同。

对应 [main.ts](scripts/19-declaration-merging/main.ts)。

```typescript
interface Stamp { code: number }
const Stamp: Stamp = { code: 19 };
console.log(Stamp.code);
// 预期输出：19
```

## 2 接口合并与同名属性

同一作用域的同名接口合并成员，所以值必须满足合并后的整体结构。非函数同名成员需要类型兼容到规则要求的相同声明类型；不能一处写 number、一处写 string，期待得到联合。

合并不负责把两个对象拼接，也不会自动补字段；下面仍由对象字面量提供 title 与 hours。

对应 [main.ts](scripts/19-declaration-merging/main.ts)。

```typescript
interface Lesson { title: string }
interface Lesson { hours: number }
const lesson: Lesson = { title: "合并", hours: 2 };
console.log(lesson.title, lesson.hours);
// 预期输出：合并 2
```

## 3 合并后的重载顺序

同名方法成员组成重载。后出现的接口声明中的重载组通常排在前面，每组内部保留顺序；单个字符串字面量参数的特化签名会前移。调用选择与用 ReturnType 提取最后签名是两种不同操作。

下面把一般输入签名与具体 count 输入签名分开声明。实际函数同时实现两条签名，特定字符串可以获得更精确的数值结果。

对应 [main.ts](scripts/19-declaration-merging/main.ts)。

```typescript
interface Reader { read(value: string): string | number }
interface Reader { read(value: "count"): number }
function read(value: "count"): number;
function read(value: string): string | number;
function read(value: string): string | number {
  return value === "count" ? 3 : value;
}
const reader: Reader = { read };
const count: number = reader.read("count");
console.log(count, reader.read("title"));
// 预期输出：3 title
```

## 4 被扩充的原模块

模块扩充（module augmentation）为已经存在的模块声明补充成员；模块名按普通导入说明符解析。原始 Gauge 类只负责保存读数，没有 double 方法。

扩充文件必须是模块，且拼写与消费者解析到的模块一致；不能把同名但实际指向别处的路径当成同一模块。

对应 [gauge.ts](scripts/19-declaration-merging/gauge.ts)。

```typescript
export class Gauge {
  constructor(public value: number) {}
}
```

## 5 类型扩充与原型实现一起加载

给类型增加一个方法名与给原型安装一个函数，是两项必须对应的工作。

gauge-double.ts 顶层 import 先建立模块身份。随后分两层补齐能力：declare module 告诉检查器 Gauge 实例具有 double 方法，原型赋值让运行中的实例真正能找到这个函数。两层缺少任何一层，都不能完成本例。

模块扩充只能补已有导出声明，不能借此添加新的顶层导出；默认导出没有可按普通导出名扩充的目标，通常应暴露具名声明或使用显式包装函数。

![模块扩充需要类型与实现同时到位。gauge-double.ts 的两段内容分别改变检查契约和运行对象。](image/illustration/19-01-augmentation-two-parts.svg)

图示说明：图仅展示对已有具名 Gauge 的扩充；声明合并本身不会执行原型赋值。

下面分别定位 declare module 和原型赋值，再查看下一节 main.ts 的副作用导入如何让两部分在一次运行中配合。

对应 [gauge-double.ts](scripts/19-declaration-merging/gauge-double.ts)。

```typescript
import { Gauge } from "./gauge.js";
declare module "./gauge.js" {
  interface Gauge { double(): number }
}
Gauge.prototype.double = function () { return this.value * 2; };
```

## 6 消费者加载扩充实现

消费者使用副作用导入执行原型赋值，再调用新方法。类型检查能看见声明还不够，运行时也必须加载扩充文件；这里每个示例以新 Node 进程运行，不依赖上次运行残留的原型。

对应 [main.ts](scripts/19-declaration-merging/main.ts)。

```typescript
import { Gauge } from "./gauge.js";
import "./gauge-double.js";
import "./global-build.js";
console.log(new Gauge(4).double(), globalThis.chapter19Build);
globalThis.chapter19Build = undefined;
// 预期输出：8 local
```

## 7 模块内部的全局扩充

declare global 允许从模块内部补充全局声明，模块中需要有 import 或 export。下面用本章专属全局名展示类型与初始化的配合，不修改标准内置对象。

声明只纳入本章项目；实际值仍由赋值提供。消费者用完后设置为 undefined，与声明的可缺省运行状态一致。

对应 [global-build.ts](scripts/19-declaration-merging/global-build.ts)。

```typescript
export {};
declare global {
  var chapter19Build: string | undefined;
}
globalThis.chapter19Build = "local";
```

## 8 只加载类型不会执行补丁

import type 让编译器读取扩充声明，却在输出中擦除导入。missing-patch.ts 因此可通过检查，但新进程没有执行原型赋值，调用会抛 TypeError。

Step 1：在 build:19 之后单独运行缺失补丁的反例。

```bash
node .build/19-declaration-merging/missing-patch.js
# 预期退出码 1，TypeError 包含 double is not a function。
```

对应 [missing-patch.ts](scripts/19-declaration-merging/missing-patch.ts)。

```typescript
import { Gauge } from "./gauge.js";
import type {} from "./gauge-double.js";
new Gauge(4).double(); // 类型导入不会执行 Gauge.prototype.double 的赋值。
```

## 9 命名空间与类、函数、枚举合并

阅读旧库时，namespace 可能用来给类增加静态成员、给函数添加属性，或给枚举附加辅助函数。与类、函数合并时，namespace 放在目标声明之后；导出的成员才能通过外部的点路径访问，未导出成员不因另一个声明同名就变成公开成员。

下例用三个小对象观察合并后的运行值。不同模块的同名 namespace 不会因此自动共享对象；不要把这种组织方式与 ESM 的模块加载混为一谈。

对应 [main.ts](scripts/19-declaration-merging/main.ts)。

```typescript
class Card { constructor(public title: string) {} }
namespace Card { export const category = "note"; }
function tag(value: string) { return `${tag.prefix}${value}`; }
namespace tag { export const prefix = "#"; }
enum Phase { Ready = 1, Done = 2 }
namespace Phase { export function label(value: Phase) { return value === Phase.Ready ? "就绪" : "完成"; } }
console.log(new Card("卡片").title, Card.category, tag("学习"), Phase.label(Phase.Done));
// 预期输出：卡片 note #学习 完成
```

## 10 不可合并的声明

类型别名不能像接口那样重复声明来增补成员，两个类声明也不能直接合成一个类；命名相同不等于具备合并资格。需要不同职责时使用不同名称、接口扩展或组合；只有真实提供实现的对象才适合用模块扩充补充类型。

下方反例同时展示属性冲突、重复别名与重复类。诊断用于定位声明关系，不能靠把返回值断言成目标类型来修复。

## 11 检查类型边界

下面的 [type-errors.ts](scripts/19-declaration-merging/type-errors.ts) 只用于检查，不执行。逐项阅读注释，修正时保留原本需求，不通过断言或关闭检查掩盖错误。

```typescript
interface Conflict { value: number }
interface Conflict { value: string } // 同名非函数成员不能改成另一类型。
type Alias = { id: number };
type Alias = { name: string }; // 类型别名不支持这种合并。
class Duplicate {}
class Duplicate {} // 类不能这样重复声明。
export {};
// 预期诊断包含：TS2717, TS2300。
```

Step 1：单独检查反例并对照错误位置与原因。

```bash
npm run errors:19
# 本章固定编译器预期退出码为 1；正常项目命令的退出码为 0。
```

## 本章小结

合并受声明种类、作用域和签名顺序约束。模块扩充要补已有具名导出，并实际加载实现；全局扩充也只提供类型依据，初始化与清理仍由程序负责。

## 练习

1. 给 Gauge 增加 triple() 的声明与实现，核对 new Gauge(4).triple() 返回 12。

2. 只移除消费者对 gauge-double.js 的值导入，保留类型导入，复现 TypeError；恢复副作用导入后重新运行。

3. 为合并的 Lesson 接口再增加可选 note，创建有 note 与无 note 的对象并检查；解释可选属性为何没有自动得到默认值。

## 参考与引用来源

- TypeScript 官方文档：[Declaration Merging：名称空间、接口、重载、合并限制、模块与全局扩充](https://www.typescriptlang.org/docs/handbook/declaration-merging.html)；[Declaration Files Deep Dive：类型与值](https://www.typescriptlang.org/docs/handbook/declaration-files/deep-dive.html)；[Modules Reference：Type-only imports、Ambient modules](https://www.typescriptlang.org/docs/handbook/modules/reference.html)；[Namespaces：导出与跨声明可见性](https://www.typescriptlang.org/docs/handbook/namespaces.html)。
- Node.js 24.11.0：[ESM：导入与执行](https://nodejs.org/download/release/v24.11.0/docs/api/esm.html)。